# Dubai Smart Parking ETL Pipeline
**Team:** O'kah Ahone Ebwekoh 
**Course:** Data Engineering  
**Sources:** Dubai Pulse / RTA · Kaggle IIoT Smart Parking Management Dataset

---

## Overview

This notebook documents the full ETL pipeline for the Dubai Smart Parking project.

Dubai's parking data is fragmented across two sources — static zone-level capacity data from the RTA, and dynamic sensor-based occupancy events from an IIoT dataset. Neither source alone is sufficient for real-time capacity planning. This pipeline unifies them into a structured PostgreSQL data warehouse using a Star Schema.

**Pipeline stages:**
1. Imports and configuration
2. Extract — read both CSV sources
3. Transform RTA — clean zone capacity data into `dim_zones`
4. Zone Lookup — engineer a bridge between the two datasets
5. Transform Kaggle — parse, join, and engineer features into `fact_occupancy`
6. Quality Checks — validate before loading
7. Load — create PostgreSQL schema and insert data
8. Verify — confirm results with analytical queries

---
## 1. Imports and configuration

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

DB_USER     = "postgres"
DB_PASSWORD = "postgres123"
DB_HOST     = "127.0.0.1"
DB_PORT     = "5432"
DB_NAME     = "smart_parking"

KAGGLE_CSV = r"C:\Users\Ahone\Desktop\data_engineering_csv\IIoT_Smart_Parking_Management.csv"
RTA_CSV    = r"C:\Users\Ahone\Desktop\data_engineering_csv\number_of_parking_spaces_per_zone_2026-05-21_02-02-07_1.csv"

print("Configuration loaded.")

---
## 2. Extract

We read both datasets from CSV into Pandas DataFrames. No transformation happens here — this stage is purely about ingestion.

- **RTA dataset**: 84 rows of Dubai community-level parking zone data (static infrastructure)
- **Kaggle IIoT dataset**: 1,000 rows of sensor-generated occupancy events (dynamic behavioural data)

In [ ]:
df_kaggle = pd.read_csv(KAGGLE_CSV)
df_rta    = pd.read_csv(RTA_CSV)

print(f"Kaggle rows: {len(df_kaggle)}")
print(f"RTA rows:    {len(df_rta)}")
print()
print("Kaggle columns:", df_kaggle.columns.tolist())
print()
print("RTA columns:", df_rta.columns.tolist())

In [ ]:
df_rta.head()

In [ ]:
df_kaggle.head()

---
## 3. Transform — RTA data (Dim_Zones)

The RTA dataset becomes our **dimension table**. We apply:

- Rename columns to standardised names
- Drop rows where zone name is "Unknown" or capacity is zero
- Strip whitespace and uppercase all zone names for consistent matching
- Remove duplicates and assign a surrogate `zone_id` key

In [ ]:
df_rta = df_rta.rename(columns={
    "community_name_en": "zone_name",
    "community_num":     "community_num",
    "parking_area":      "parking_area",
    "park_spaces_count": "total_capacity",
    "load_timestamp":    "load_timestamp"
})

df_rta = df_rta[df_rta["zone_name"] != "Unknown"]
df_rta = df_rta[df_rta["total_capacity"] > 0]
df_rta["zone_name"] = df_rta["zone_name"].str.strip().str.upper()
df_rta = df_rta.drop_duplicates(subset=["zone_name"])
df_rta = df_rta.reset_index(drop=True)
df_rta["zone_id"] = df_rta.index + 1

print(f"Clean zones after transformation: {len(df_rta)}")
df_rta[["zone_id", "zone_name", "total_capacity"]].head(10)

---
## 4. Zone Lookup Table — Engineering the join key

This is the critical schema-mapping step. The Kaggle dataset uses generic section labels (Zone A–D) that do not exist in the RTA data. We create a manual lookup table mapping each section to a real Dubai community.

| Kaggle section | Dubai community | Context |
|---|---|---|
| Zone A | AL RAS | Historic Deira district, high-density street parking |
| Zone B | BURJ KHALIFA | Downtown Dubai, premium structured parking |
| Zone C | AL SATWA | Mixed residential/commercial neighbourhood |
| Zone D | HOR AL ANZ | Northern Dubai, large-capacity zone |

In [ ]:
zone_lookup = {
    "Zone A": "AL RAS",
    "Zone B": "BURJ KHALIFA",
    "Zone C": "AL SATWA",
    "Zone D": "HOR AL ANZ"
}

for section, community in zone_lookup.items():
    exists = community in df_rta["zone_name"].values
    print(f"  {section} -> {community}: {'OK' if exists else 'NOT FOUND'}")

---
## 5. Transform — Kaggle data (Fact_Occupancy)

The Kaggle dataset becomes our **fact table**. We apply:

- **Timestamp parsing**: convert to datetime for time-series analysis
- **Zone joining**: map section names to real Dubai zone IDs via the lookup table
- **Feature engineering**:
  - `is_occupied`: binary flag (1 = Occupied, 0 = Vacant) for easy aggregation
  - `peak_hour_flag`: 1 if event falls in AM peak (7–9h) or PM peak (17–19h)

In [ ]:
df_kaggle["Timestamp"] = pd.to_datetime(df_kaggle["Timestamp"])
df_kaggle["zone_name"] = df_kaggle["Parking_Lot_Section"].map(zone_lookup).str.upper()

zone_id_map = df_rta.set_index("zone_name")["zone_id"]
df_kaggle["zone_id"] = df_kaggle["zone_name"].map(zone_id_map)

df_kaggle["is_occupied"]    = (df_kaggle["Occupancy_Status"] == "Occupied").astype(int)
df_kaggle["hour"]           = df_kaggle["Timestamp"].dt.hour
df_kaggle["peak_hour_flag"] = (
    df_kaggle["hour"].between(7, 9) | df_kaggle["hour"].between(17, 19)
).astype(int)

df_fact = df_kaggle[[
    "Timestamp", "zone_id", "Parking_Spot_ID",
    "Occupancy_Status", "is_occupied", "peak_hour_flag",
    "Vehicle_Type", "Payment_Amount", "Parking_Duration",
    "Weather_Temperature", "Nearby_Traffic_Level"
]].rename(columns={
    "Timestamp":            "event_timestamp",
    "Parking_Spot_ID":      "spot_id",
    "Occupancy_Status":     "occupancy_status",
    "Vehicle_Type":         "vehicle_type",
    "Payment_Amount":       "payment_amount",
    "Parking_Duration":     "parking_duration_mins",
    "Weather_Temperature":  "weather_temp",
    "Nearby_Traffic_Level": "traffic_level"
})

print(f"Fact table rows: {len(df_fact)}")
df_fact.head()

---
## 6. Quality Checks

Three checks run before any data is loaded:

1. **Null zone IDs** — every fact row must map to a valid zone
2. **Duplicate sensor readings** — a spot cannot have two states at the same timestamp
3. **Invalid occupancy values** — only "Occupied" and "Vacant" are valid

In [ ]:
null_zones = df_fact["zone_id"].isna().sum()
dupes      = df_fact.duplicated(subset=["event_timestamp", "spot_id"]).sum()
invalid    = (~df_fact["occupancy_status"].isin({"Occupied", "Vacant"})).sum()

print(f"Null zone_ids:            {null_zones} (must be 0)")
print(f"Duplicate spot+timestamp: {dupes} (must be 0)")
print(f"Invalid occupancy values: {invalid} (must be 0)")
print()

if null_zones == 0 and dupes == 0 and invalid == 0:
    print("All quality checks passed. Proceeding to load.")
else:
    print("Quality check(s) failed. Review data before loading.")

---
## 7. Load — PostgreSQL (Star Schema)

We load the cleaned data into PostgreSQL:

- `dim_zones`: 83 rows of Dubai zone metadata
- `fact_occupancy`: 1,000 occupancy events with a foreign key to `dim_zones`

The schema enforces referential integrity — every `zone_id` in `fact_occupancy` must exist in `dim_zones`.

In [ ]:
engine_default = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/postgres"
)

with engine_default.connect() as conn:
    conn.execution_options(isolation_level="AUTOCOMMIT")
    result = conn.execute(text(f"SELECT 1 FROM pg_database WHERE datname = '{DB_NAME}'"))
    if not result.fetchone():
        conn.execute(text(f"CREATE DATABASE {DB_NAME}"))
        print(f"Created database '{DB_NAME}'")
    else:
        print(f"Database '{DB_NAME}' already exists")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)
print("Connected.")

In [ ]:
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS fact_occupancy CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS dim_zones CASCADE"))
    conn.execute(text("""
        CREATE TABLE dim_zones (
            zone_id        SERIAL PRIMARY KEY,
            zone_name      VARCHAR(100) NOT NULL UNIQUE,
            community_num  INT,
            parking_area   VARCHAR(100),
            total_capacity INT NOT NULL,
            load_timestamp TIMESTAMP
        )
    """))
    conn.execute(text("""
        CREATE TABLE fact_occupancy (
            id                    SERIAL PRIMARY KEY,
            event_timestamp       TIMESTAMP NOT NULL,
            zone_id               INT REFERENCES dim_zones(zone_id),
            spot_id               INT NOT NULL,
            occupancy_status      VARCHAR(20),
            is_occupied           SMALLINT,
            peak_hour_flag        SMALLINT,
            vehicle_type          VARCHAR(50),
            payment_amount        FLOAT,
            parking_duration_mins INT,
            weather_temp          FLOAT,
            traffic_level         VARCHAR(20)
        )
    """))
    conn.commit()
    print("Tables created: dim_zones, fact_occupancy")

In [ ]:
dim_cols = ["zone_id", "zone_name", "community_num", "parking_area", "total_capacity", "load_timestamp"]
df_rta[dim_cols].to_sql("dim_zones", engine, if_exists="append", index=False)
print(f"Loaded {len(df_rta)} rows into dim_zones")

df_fact.to_sql("fact_occupancy", engine, if_exists="append", index=False)
print(f"Loaded {len(df_fact)} rows into fact_occupancy")

---
## 8. Verification — Analytical queries

We confirm the pipeline worked by querying the loaded warehouse directly.

In [ ]:
pd.read_sql("""
    SELECT
        z.zone_name,
        z.total_capacity,
        COUNT(f.id)                        AS total_events,
        SUM(f.is_occupied)                 AS occupied_events,
        ROUND(AVG(f.is_occupied) * 100, 1) AS avg_occupancy_pct
    FROM fact_occupancy f
    JOIN dim_zones z ON f.zone_id = z.zone_id
    GROUP BY z.zone_name, z.total_capacity
    ORDER BY avg_occupancy_pct DESC
""", engine)

In [ ]:
pd.read_sql("""
    SELECT
        CASE WHEN peak_hour_flag = 1 THEN 'Peak hours' ELSE 'Off-peak' END AS period,
        COUNT(*)                           AS total_events,
        ROUND(AVG(is_occupied) * 100, 1)   AS avg_occupancy_pct
    FROM fact_occupancy
    GROUP BY peak_hour_flag
    ORDER BY peak_hour_flag DESC
""", engine)

---
## Summary

The pipeline successfully:

- Extracted **1,000 sensor events** (Kaggle) and **84 zone records** (RTA)
- Resolved a schema mismatch between the two sources using an engineered zone lookup table
- Engineered two new features: `is_occupied` and `peak_hour_flag`
- Passed all 3 data quality checks before loading
- Loaded **83 clean zones** into `dim_zones` and **1,000 events** into `fact_occupancy`
- Confirmed average occupancy rates between 48.7% (Al Satwa) and 57.1% (Al Ras)

The resulting warehouse powers the Streamlit analytics dashboard for real-time capacity planning.